# Lab 02: Training a Custom Model


**Objective of this lab**: training a small custom model on the Tiny-ImageNet dataset.

## Dataset preparation

In [ ]:
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip -d tiny-imagenet

We need to adjust the format of the val split of the dataset to be used with ImageFolder.

In [ ]:
import os
import shutil

with open('tiny-imagenet/tiny-imagenet-200/val/val_annotations.txt') as f:
    for line in f:
        fn, cls, *_ = line.split('\t')
        os.makedirs(f'tiny-imagenet/tiny-imagenet-200/val/{cls}', exist_ok=True)

        shutil.copyfile(f'tiny-imagenet/tiny-imagenet-200/val/images/{fn}', f'tiny-imagenet/tiny-imagenet-200/val/{cls}/{fn}')

shutil.rmtree('tiny-imagenet/tiny-imagenet-200/val/images')

- ImageFolder expects this structure:  
val/  
│  
├── class_1/
│   ├── img1.jpg
│   ├── img2.jpg
│
├── class_2/
│   ├── img3.jpg
│   ├── img4.jpg

fn, cls, *_ = line.split('\t') --> This splits the line by tab and returns (example):
fn  = "val_123.JPEG"
cls = "n01629819"

- create the class folder: os.makedirs(f'.../val/{cls}', exist_ok=True)
- the images from the validation folder to its class
shutil.copyfile(  
    'val/images/val_123.JPEG',  
    'val/n01629819/val_123.JPEG'  
)  


- So after this block the validation becomes:
val/
│
├── n01443537/
│   ├── val_0.JPEG
│   ├── val_44.JPEG
│
├── n01629819/
│   ├── val_1.JPEG
│   ├── val_78.JPEG


In [ ]:
from torchvision.datasets import ImageFolder
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((224, 224)),  # Resize to fit the input dimensions of the network
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# root/{classX}/x001.jpg

tiny_imagenet_dataset_train = ImageFolder(root='tiny-imagenet/tiny-imagenet-200/train', transform=transform)
tiny_imagenet_dataset_val = ImageFolder(root='tiny-imagenet/tiny-imagenet-200/val', transform=transform)

In [ ]:
print(f"Length of train dataset: {len(tiny_imagenet_dataset_train)}")
print(f"Length of val dataset: {len(tiny_imagenet_dataset_val)}")

# The following code also checks the number of samples per class
# from collections import Counter

# class_counts = Counter([target for _, target in tiny_imagenet_dataset_val])
# for class_label, count in class_counts.items():
#   print(f"Class {class_label}: {count} entries")


In [ ]:
train_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_train, batch_size=32, shuffle=True, num_workers=8)
val_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_val, batch_size=32, shuffle=False)

# Convolutional Neural Networks CNN
First thing first, images has shapes (channel, height, width)

- What is a channel?
  - A channel is just one grid of numbers.

- Example 4×4 image:
``` text
[ 10 20 30 40 ]  
[ 15 25 35 45 ]  
[ 20 30 40 50 ]  
[ 25 35 45 55 ]  
```
(1, 4, 4)  

> RGB   
- RGB images have 3 channels = 1 RGB picture is obtained by the SOVRAPPOSIZIONE = **STACK** OF 3 MATRIXES: ONE FOR THE RED, ONE FOR THE GREEN AND ONE FOR THE BLUE.
- Each channel is a separate matrix.
  - Red channel
  - Green channel
  - Blue channel

Example:
``` text
R =  
[10 20]  
[30 40]

G =  
[ 5 15]  
[25 35]  

B =  
[ 2 12]  
[22 32]  
```

So the shape is: (3, 2, 2) --> 3 channels ; 2×2 image
> **So an image is really a stack of matrices --> in the RGB case an image is a stack of 3 matrixes.**.  
> OCCHIO EH, **STACK** NOT SUM.

- In this dataset case we have: (3, 64, 64) --> **each picture is the stack of 3 64x64 matrixes**


---


# .Conv2d(in_channels, out_channels, kernel_size, stride, padding)
self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=1)  
- core layer used in Convolutional Neural Networks (CNNs)
- A convolutional layer takes an input image and applies small filters that scan the image.
  - Think of a filter like this:
``` text
3x3 kernel  
[ w1 w2 w3 ]  
[ w4 w5 w6 ]  
[ w7 w8 w9 ]
```

- nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)

- Conv2d is just a layer that looks at small patches of the image and learns to detect patterns.
- Patterns like:
  - edges
  - lines
  - corners
  - textures

- for example you have a 64 x 64 pixels image
- Instead of looking at the whole image at once, the network looks at small squares like: 3x3 pixels
- Like:
``` text
[ 12  25  18 ]  
[ 20  40  35 ]  
[ 10  15  30 ]  
```

# What is the kernel (filter)? kernel_size = 3
- The network learns a small 3×3 detector like:
``` text
[ 1  0 -1 ]  
[ 1  0 -1 ]  
[ 1  0 -1 ]  
```
- This one might detect vertical edges.
- So the network slides this detector over the image and asks:
- “Does this patch look like the pattern I’m searching for?”
- Input channels = 3 --> because RGB
- **Output channels = 64** --> This means the layer learns 64 different pattern detectors.

- Example:
  - detector 1 → vertical edge
  - detector 2 → horizontal edge
  - detector 3 → color gradient
  - detector 4 → texture
  - ...
  - detector 64 → some weird pattern

- Padding = 1 --> “Add a thin border around the image so the size stays the same.”
- Stride = 1 --> move the detector 1 pixel at a time --> TO SCAN EVERY LOCATION


### **Conv2d applies multiple learnable filters that slide across the image to extract visual features, producing a set of feature maps.**


> # EXAMPLE
``` text
input (1 channel)  
[ 1 2 3 4 ]  
[ 5 6 7 8 ]  
[ 9 1 2 3 ]  
[ 4 5 6 7 ]  
```

- Now we use a 3×3 kernel.
  - Example kernel:  
``` text
[ 1 0 0 ]  
[ 0 1 0 ]  
[ 0 0 1 ]
```  
  - This kernel will slide across the image.

> Take the first 3×3 patch:
``` text
[1 2 3]  
[5 6 7]  
[9 1 2]
```  

- Multiply element-by-element with the kernel:
``` text
[1 2 3]   [1 0 0]  
[5 6 7] * [0 1 0]  
[9 1 2]   [0 0 1]  
```

- Now sum everything:
``` text
1*1 + 2*0 + 3*0
+5*0 + 6*1 + 7*0
+9*0 + 1*0 + 2*1
```

- result: 1 + 6 + 2 = 9
- So the first pixel of the output is: 9

> Move the kernel right
- next patch:
``` text
[2 3 4]
[6 7 8]
[1 2 3]
```
- Resulting pixel: 2 + 7 + 3 = 12
- The output accumulates all the results: [ 9 12 ... ]

> AFTER SCANNING THE WHOLE IMAGE THE RESULT IS

``` text
output
[ 9 12 ]
[11 14 ]
```

- So convolution is literally:
  - slide kernel
  - multiply
  - sum
  - write number

> # Output size changed
- Input: 4x4
- Kernel: 3x3
- without the padding the output becomes: 2×2

> adding padding=1
- it adds a border, like:
``` text
0 0 0 0 0
0 1 2 3 4
0 5 6 7 8
0 9 1 2 3
0 4 5 6 7
0 0 0 0 0
```

- so the result is: output = 4×4
- Same size as input.

> # What if we use more than one Kernel?
- In our example we use just one Kernel with shape 3x3
- BUT if instead of one kernel, the network uses 64 kernels:
  - kernel_1
  - kernel_2
  - kernel_3
  - ...
  - kernel_64

- Each kernel produces its own output image.
  - kernel_1 → output image 1
  - kernel_2 → output image 2
  - ...
  - kernel_64 → output image 64
- SO the final output image is the STACK of all these 64 matrixes: (64, height, width)


> # INPUT - OUTPUT SHAPE WORKFLOW
- input: (3, 64, 64)
- passes though: Conv2d(3,64,3,padding=1)
- Output: (64, 64, 64)

---

> #### **So with nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=1) I'm rewriting an input image as the stack of 64 different images? Why would I use it? What's its purpose What's the advantage in reshaping my picture as the stack of 64 different images?**
- conceptually we are turning (3, H, W) --> (64, H, W)
- which is 64 images stacked together.
  - But they are not normal images anymore.
  - They are **feature maps** : each one highlights something specific in the image.

- So if we have:
``` text
input (1 channel)
[ 1 2 3 4 ]
[ 5 6 7 8 ]
[ 9 1 2 3 ]
[ 4 5 6 7 ]
```

- Now imagine a kernel that detects vertical edges:
``` text
[ 1 0 -1 ]
[ 1 0 -1 ]
[ 1 0 -1 ]
```

- If you slide it across the image, the output might look like:
``` text
vertical edge map

[ 5  2  0  3 ]
[ 4  1 -1  2 ]
[ 6  3  0  1 ]
[ 5  2 -2  0 ]
```

- Bright values mean = "there is a vertical edge here"
- That output is one feature map.

Now:
- Instead of one kernel, we use 64 kernels.
- Each kernel looks for a different pattern.
- Each one produces its own map --> 64 maps
- We rewrite the input picture as a stack of these feature maps:
``` text
(feature_map_1
 feature_map_2
 feature_map_3
 ...
 feature_map_64)
```

# SO WHY DID WE EXPRESS THE ORIGINAL PICTURE AS A (64, H, W)?
> Because the network rewrites the image in terms of useful patterns.  
> Instead of raw pixels: R G B values.
> the representation becomes:  
- where edges are
- where textures are
- where shapes start
- where corners appear
> So the network transforms the image into a much more informative representation.  



---

> # POOLING
- After the convolution layers we get something like: (64, 224, 224)
- This is humongous, big as fuck ... we need to reduce it --> **pooling**
- Kind of a dimensionality reduction but only for height and width, keeping the same number of channels
  - Pooling allows to: (64, 224, 224) --> (64, 112, 112)


> EXAMPLE
``` text
[ 1  3  2  4 ]
[ 5  6  1  2 ]
[ 0  2  3  1 ]
[ 4  1  2  0 ]
```

- Now apply **MaxPool** with 2×2 window.
- We divide the matrix into 2×2 blocks.
``` text
[1 3]
[5 6]
Maximum = 6

[2 4]
[1 2]
Maximum = 4

[0 2]
[4 1]
Maximum = 4

[3 1]
[2 0]
Maximum = 3
```

- therefore the new matrix:
``` text
[6 4]
[4 3]
```

- Pooling just allowed to: 4×4 → 2×2

``` python
pool = nn.MaxPool2d(kernel_size=2, stride=2)
y = pool(x)
```

> Other pooling methods:
- Average Pooling
  - Instead of the max, it takes the average value.
  - pool = nn.AvgPool2d(kernel_size=2, stride=2)
    - stride = how many pixels the operation moves each step when scanning the image

- Global Average Pooling
  - Instead of pooling small blocks, you average the entire feature map.
  - pool = nn.AdaptiveAvgPool2d((1,1))

- Adaptive Pooling
  - Used when you want the output to be a specific size regardless of input size.
  - nn.AdaptiveMaxPool2d((7,7))


---

# WORKFLOW
- conv --> input (3, 224, 224) --> output (64, 224, 224)
- relu
- conv --> input (64, 224, 224) --> output (128, 224, 224)
- relu
- pool --> (128, 112, 112)

- conv --> input (128, 112, 112) --> (256, 112, 112)
- relu
- pool --> to (256, ?, ?) --> **INUTILE**

- What do I pool to?
  - Let's take for example: (256, 50, 50)  
  - We know the final layer is linear and it's 'nn.Linear(256, 200)' (the professor wrote it)
  - SO we need to flatten (256, 50, 50) --> 256 * 50 * 50 = 640.000
  - and start reducing it's dimensionality, like:
nn.Linear(640.000, 200.00)
nn.Linear(200.00, 100.000)
ecc.

- Is this a good idea? **NO** --> absurdly expensive
- $\text{params} = \text{in_features} \times \text{out_features} + \text{bias}$ --> ≈ 640.000 * 200.00 = 1,28×10¹⁰ features
- That’s completely insane.


> **we need the flatten version to be 256**

---

# Use **Global Average Pooling**
Workflow:

Input image  
- (3, 224, 224)  
↓  
Conv1  
- Conv2d(3 → 64)  
- Output: (64, 224, 224)  
↓  
ReLU  
↓  
Conv2  
- Conv2d(64 → 128)  
- Output: (128, 224, 224)  
↓  
ReLU  
↓  
MaxPool  
- Reduce spatial size by half  
- (128, 224, 224) → (128, 112, 112)  
↓  
Conv3  
- Conv2d(128 → 256)  
- Output: (256, 112, 112)  
↓  
ReLU  
↓  
**Global Average Pooling**  
- Compress each feature map to one number  
(256, 112, 112)  
↓  
(256, 1, 1)  
- **Each of the 256 feature maps becomes a single value that represents how strongly that feature appears in the image.**

↓  
Flatten = (256)  
↓  
Linear layer  
- Linear(256 → 200)  




In [ ]:
import torch
from torch import nn

# Define the custom neural network
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()

        # Define layers of the neural network
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)       # input (3, 224, 224) --> output (64, 224, 224)
                                                                      # then --> ReLU
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)     # input (64, 224, 224) --> output (128, 224, 224)
                                                                      # then --> ReLU
                                                                      # then Maxpooling: (128, 224, 224) --> (128, 112, 112)
        # ReLu
        self.relu = nn.ReLU()

        # MaxPooling --> MAH, serve davvero???
        self.Maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Convolution block 2: (128, 112, 112) --> (256, 112, 112)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)    # input (128, 112, 112) --> output (256, 112, 112)
                                                                      # then --> ReLU
                                                                      # then --> Global Average Pooling: (256, 112, 112) --> (256, 1, 1)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))

        # final layer
        self.linear1 = nn.Linear(256, 200) # 200 is the number of classes in TinyImageNet   # input (256) --> output (200)





    def forward(self, x):
        # Define forward pass
        x = self.conv1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.Maxpool(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.global_avg_pool(x)

        # flatten (256, 1, 1) --> (256)
        x = torch.flatten(x, 1)


        x = self.linear1(x)

        return x

- Until now we ONLY considered one picture, but we are considering more than one picture of course, we are considering batch_size pictures
- So the actual shapes are:

``` text
conv1              -> (batch_size, 64, 224, 224)
ReLU               -> (batch_size, 64, 224, 224)

conv2              -> (batch_size, 128, 224, 224)
ReLU               -> (batch_size, 128, 224, 224)
MaxPool            -> (batch_size, 128, 112, 112)

conv3              -> (batch_size, 256, 112, 112)
ReLU               -> (batch_size, 256, 112, 112)
GlobalAvgPool      -> (batch_size, 256, 1, 1)

flatten            -> (batch_size, 256)
linear1            -> (batch_size, 200)
```

---

# Train


In [ ]:
def train(epoch, model, train_loader, loss_fn, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # I have a mac --> no cuda
        inputs = inputs.to(device)
        targets = targets.to(device)


        # training heart
        optimizer.zero_grad()
        predictions = model(inputs)
        loss = loss_fn(predictions, targets)
        loss.backward()
        optimizer.step()


        running_loss += loss.item()
        predicted = predictions.argmax(dim=1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        # correct += (predicted == targets).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total
    print(f'Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%')

In [ ]:
# Validation loop
def validate(model, val_loader, loss_fn):
    model.eval()
    val_loss = 0

    correct, total = 0, 0

    with torch.no_grad():
        for batch_idx,(inputs, targets) in enumerate(val_loader):
          # I have a mac --> no cuda
          inputs = inputs.to(device)
          targets = targets.to(device)


          # no gradients, only predict and look how much the trained model was wrong
          predictions = model(inputs)
          loss = loss_fn(predictions, targets)


          val_loss += loss.item()
          predicted = predictions.argmax(dim=1)
          total += targets.size(0)
          correct += predicted.eq(targets).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100. * correct / total

    print(f'Validation Loss: {val_loss:.6f} Acc: {val_accuracy:.2f}%')
    return val_accuracy

## Putting everything together

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = CustomNet().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)


best_acc = 0

# Run the training process for {num_epochs} epochs
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train(epoch, model, train_loader, loss_fn, optimizer)

    # At the end of each training iteration, perform a validation step
    val_accuracy = validate(model, val_loader, loss_fn)

    # Best validation accuracy
    best_acc = max(best_acc, val_accuracy)


print(f'Best validation accuracy: {best_acc:.2f}%')


---

# CRITIQUE & IMPROVEMENT
ChatGTP critique is:
*"it's a bit unusual as the first two convs keep the full 224 x 224 resolution."*

	•	more computation
	•	more memory usage
	•	not much downsampling early on
So it works, but it is a bit “heavy” spatially.


Could be better to do:
- conv -> relu -> pool
- conv -> relu -> pool
- conv -> relu -> global avg pool
- linear


# Whole code

In [ ]:
import os
import shutil

with open('tiny-imagenet/tiny-imagenet-200/val/val_annotations.txt') as f:
    for line in f:
        fn, cls, *_ = line.split('\t')
        os.makedirs(f'tiny-imagenet/tiny-imagenet-200/val/{cls}', exist_ok=True)

        shutil.copyfile(f'tiny-imagenet/tiny-imagenet-200/val/images/{fn}', f'tiny-imagenet/tiny-imagenet-200/val/{cls}/{fn}')

shutil.rmtree('tiny-imagenet/tiny-imagenet-200/val/images')

In [ ]:
import torch
from torch import nn
from torchvision.datasets import ImageFolder
import torchvision.transforms as T


def train_val_dataset():
  transform = T.Compose([
      T.Resize((224, 224)),  # Resize to fit the input dimensions of the network
      T.ToTensor(),
      T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
      ])

  # root/{classX}/x001.jpg
  tiny_imagenet_dataset_train = ImageFolder(root='tiny-imagenet/tiny-imagenet-200/train', transform=transform)
  tiny_imagenet_dataset_val = ImageFolder(root='tiny-imagenet/tiny-imagenet-200/val', transform=transform)

  return tiny_imagenet_dataset_train, tiny_imagenet_dataset_val


# ------------------------------
# Neural Network
# ------------------------------
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()

        # Convolution block 1
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)       # (3,224,224) -> (64,224,224)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)     # (64,224,224) -> (128,224,224)

        # Activation
        self.relu = nn.ReLU()

        # MaxPooling
        self.Maxpool = nn.MaxPool2d(kernel_size=2, stride=2)          # (128,224,224) -> (128,112,112)

        # Convolution block 2
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)    # (128,112,112) -> (256,112,112)

        # Global Average Pooling
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))             # (256,112,112) -> (256,1,1)

        # Final classification layer
        self.linear1 = nn.Linear(256, 200)                             # TinyImageNet → 200 classes


    def forward(self, x):

        x = self.conv1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.Maxpool(x)

        x = self.conv3(x)
        x = self.relu(x)

        x = self.global_avg_pool(x)

        # Flatten (batch,256,1,1) → (batch,256)
        x = torch.flatten(x, 1)

        x = self.linear1(x)

        return x




# ------------------------------
# Training Loop
# ------------------------------
def train(epoch, model, train_loader, loss_fn, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):

        inputs = inputs.to(device)
        targets = targets.to(device)


        optimizer.zero_grad()
        predictions = model(inputs)
        loss = loss_fn(predictions, targets)
        loss.backward()
        optimizer.step()


        running_loss += loss.item()

        predicted = predictions.argmax(dim=1)

        total += targets.size(0)

        correct += predicted.eq(targets).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total

    print(f"Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%")


# ------------------------------
# Validation Loop
# ------------------------------
def validate(model, val_loader, loss_fn):

    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):

            inputs = inputs.to(device)
            targets = targets.to(device)


            predictions = model(inputs)
            loss = loss_fn(predictions, targets)


            val_loss += loss.item()
            predicted = predictions.argmax(dim=1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100. * correct / total

    print(f"Validation Loss: {val_loss:.6f} Acc: {val_accuracy:.2f}%")

    return val_accuracy






# ------------------------------
# Entry Point
# ------------------------------
if __name__ == "__main__":
  tiny_imagenet_dataset_train, tiny_imagenet_dataset_val = train_val_dataset()
  train_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_train, batch_size=32, shuffle=True, num_workers=8)
  val_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_val, batch_size=32, shuffle=False)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = CustomNet().to(device)
  loss_fn = nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

  best_acc = 0

  num_epochs = 10

  for epoch in range(1, num_epochs + 1):
    train(epoch, model, train_loader, loss_fn, optimizer)

    val_accuracy = validate(model, val_loader, loss_fn)

    print(f"Epoch: {epoch} --> accuracy: {val_accuracy}")
    best_acc = max(best_acc, val_accuracy)

  print(f"\n\nBest validation accuracy: {best_acc:.2f}%")

Train Epoch: 1 Loss: 5.241613 Acc: 1.19%
Validation Loss: 5.102421 Acc: 1.62%
Epoch: 1 --> accuracy: 1.62
Train Epoch: 2 Loss: 5.065296 Acc: 2.05%
Validation Loss: 5.014392 Acc: 2.61%
Epoch: 2 --> accuracy: 2.61
